<a href="https://colab.research.google.com/github/jackevansadl/chem2002/blob/main/C2/01_Atoms_to_Orbitals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: From Atomic Orbitals to Molecular Orbitals 🧲

Welcome to the first notebook of the C2 workshop on **chemical bonding**! In C1 you treated atoms as simple spheres. Now we go inside the atom and ask *why* atoms bond at all.

The answer is **Molecular Orbital (MO) theory**: when atoms come together, their **atomic orbitals (AOs)** combine to form new **molecular orbitals** that are spread over the whole molecule. In this notebook you will:

* visualise the building blocks — *s* and *p* atomic orbitals,
* combine two hydrogen 1*s* orbitals into a **bonding** and an **antibonding** molecular orbital,
* see how the picture changes for a **polar** molecule (carbon monoxide).

We use **PySCF**, a quantum-chemistry program that solves the Schroedinger equation approximately, and **py3Dmol** to view the orbitals in 3D.

In [ ]:
!pip install -q pyscf py3Dmol rdkit ase

In [ ]:
# Helper functions for visualising orbitals with py3Dmol
import numpy as np
import py3Dmol
from pyscf import gto, scf, dft
from pyscf.tools import cubegen

BOHR = 0.529177  # Angstrom per Bohr

def mol_to_xyz(mol):
    """Return an XYZ string (in Angstrom) for a PySCF molecule."""
    coords = mol.atom_coords() * BOHR
    lines = [str(mol.natm), ""]
    for i in range(mol.natm):
        x, y, z = coords[i]
        lines.append(f"{mol.atom_symbol(i)} {x:.4f} {y:.4f} {z:.4f}")
    return chr(10).join(lines)

def show_orbital(mol, coeff, isoval=0.03):
    """Render a molecular orbital (blue = +, red = -) over the molecule."""
    cubegen.orbital(mol, "/tmp/orb.cube", coeff)
    cube = open("/tmp/orb.cube").read()
    v = py3Dmol.view(width=420, height=380)
    v.addModel(mol_to_xyz(mol), "xyz")
    v.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    v.addVolumetricData(cube, "cube", {"isoval":  isoval, "color": "blue", "opacity": 0.85})
    v.addVolumetricData(cube, "cube", {"isoval": -isoval, "color": "red",  "opacity": 0.85})
    v.zoomTo()
    return v.show()

def show_ao(mol, ao_index, isoval=0.05):
    """Render a single atomic orbital (basis function) by its index."""
    coeff = np.zeros(mol.nao)
    coeff[ao_index] = 1.0
    return show_orbital(mol, coeff, isoval)

-----
## 1.1 The Building Blocks: Atomic Orbitals

Atomic orbitals are the regions where an electron is likely to be found around a single atom. The two types we need for this workshop are:

* **s orbitals** — spherical, no preferred direction.
* **p orbitals** — dumbbell-shaped, pointing along the *x*, *y* or *z* axis. A p orbital has a **positive lobe (blue)** and a **negative lobe (red)**; the sign is the *phase* of the electron wave, which is the key to bonding.

Let's look at the 1*s* orbital of hydrogen and a 2*p* orbital of carbon.

In [ ]:
# A hydrogen 1s orbital (spherical)
H = gto.M(atom="H 0 0 0", basis="6-31g", spin=1, verbose=0)
show_ao(H, 0)

In [ ]:
# A carbon 2p_z orbital (dumbbell: one blue lobe, one red lobe)
C = gto.M(atom="C 0 0 0", basis="6-31g*", spin=2, verbose=0)
labels = C.ao_labels()
pz = [i for i, l in enumerate(labels) if "2pz" in l][0]
show_ao(C, pz)

-----
## 1.2 Making a Bond: H + H -> H2

When two hydrogen atoms approach, their 1*s* orbitals overlap and combine in **two** ways:

* **In phase** (both blue): electron density builds up *between* the nuclei -> a **bonding** orbital (lower energy, labelled $\sigma$).
* **Out of phase** (one blue, one red): a **node** appears between the nuclei, pushing density away -> an **antibonding** orbital (higher energy, labelled $\sigma^*$).

H2 has 2 electrons, which fill the bonding $\sigma$ orbital (the **HOMO** = Highest Occupied MO). The empty antibonding $\sigma^*$ orbital is the **LUMO** (Lowest Unoccupied MO).

Let's calculate H2 and view both orbitals.

In [ ]:
# Calculate H2 at its equilibrium bond length (0.74 Angstrom)
h2 = gto.M(atom="H 0 0 0; H 0 0 0.74", basis="6-31g", verbose=0)
mf = scf.RHF(h2).run()

n_occ = h2.nelectron // 2          # number of doubly-occupied orbitals
homo, lumo = n_occ - 1, n_occ
print(f"HOMO (sigma)  energy: {mf.mo_energy[homo]*27.211:6.2f} eV")
print(f"LUMO (sigma*) energy: {mf.mo_energy[lumo]*27.211:6.2f} eV")

In [ ]:
# The bonding sigma MO (HOMO): density builds up BETWEEN the atoms
show_orbital(h2, mf.mo_coeff[:, homo])

In [ ]:
# The antibonding sigma* MO (LUMO): a node (colour change) BETWEEN the atoms
show_orbital(h2, mf.mo_coeff[:, lumo])

-----
## 1.3 A Polar Bond: Carbon Monoxide (CO)

In H2 the two atoms are identical, so the orbitals are shared evenly. In a **heteronuclear** molecule like **CO**, the two atoms are different, so the molecular orbitals are *lopsided* — they sit more on one atom than the other. This is the origin of **bond polarity**.

Let's look at the HOMO of CO, which is a lone pair pointing away from the carbon.

In [ ]:
# Carbon monoxide
co = gto.M(atom="C 0 0 0; O 0 0 1.128", basis="6-31g*", verbose=0)
mf_co = scf.RHF(co).run()
n_occ = co.nelectron // 2
print(f"CO HOMO energy: {mf_co.mo_energy[n_occ-1]*27.211:.2f} eV")

# HOMO of CO (notice it is not symmetric between the two atoms)
show_orbital(co, mf_co.mo_coeff[:, n_occ - 1])

-----
### **📋 Reporting Task 1**

The molecule **He$_2$** would have **4** electrons: 2 in the bonding $\sigma$ orbital *and* 2 in the antibonding $\sigma^*$ orbital.

The **bond order** tells us how many net bonds hold a molecule together:

$$\text{bond order} = \tfrac{1}{2}\,(\text{bonding electrons} - \text{antibonding electrons})$$

Complete the code below to:
1. Calculate He$_2$ (use a He--He distance of 3.0 Angstrom).
2. Visualise its bonding ($\sigma$) and antibonding ($\sigma^*$) orbitals.
3. In your workbook: work out the **bond order of He$_2$** from the formula above, and explain **why He$_2$ is not a stable molecule** even though both atoms have filled 1*s* shells.

In [ ]:
# Build He2 (hint: copy the H2 cell and change the atoms)
he2 = gto.M(atom="He 0 0 0; He 0 0 3.0", basis="6-31g", verbose=0)

# Run a Hartree-Fock calculation


# Identify and visualise the bonding (sigma) orbital


# Identify and visualise the antibonding (sigma*) orbital

-----
### 🚀 Optional Bonus: a modern machine-learning functional (Microsoft *Skala*)

The calculations above use standard quantum-chemistry methods. In 2025 Microsoft Research released **Skala**, a *deep-learning* exchange-correlation functional that plugs straight into PySCF. It is more accurate for energies, but heavier to install (it downloads a neural-network model).

If you'd like to try it, **uncomment** the lines below and run the cell. (This is purely optional and not needed for any reporting task.)

In [ ]:
# Optional: try the Microsoft Skala ML-DFT functional on H2
# !pip install -q skala
# from skala.pyscf import SkalaKS
# ks = SkalaKS(h2, xc="skala-1.1")
# ks.kernel()
# print("Skala total energy:", ks.e_tot, "Hartree")

-----
## Next Steps

You've seen how atomic orbitals combine into **bonding** and **antibonding** molecular orbitals, and how the picture skews for polar bonds.

➡️ Continue to **`02_MO_Diagrams_N2_O2.ipynb`**, where we build full **MO energy-level diagrams** for N$_2$ and O$_2$ — and discover why O$_2$ is magnetic, something simple Lewis structures cannot explain.